# DenseUAV heads, paper Table VI setup: Global vs FSRA vs LPN (Kaggle, GPU T4 x2)
Same notebook as `baseline02.ipynb`, but set up the way the paper ran Table VI instead of the repo's final setup in `train_test_local.sh`:
- **Losses:** CE + `HardMiningTripletLoss`, **no KL**. The paper's Global row (80.18 R@1 / 84.39 SDM@1) is identical to Table VII's "CE + Hard-Mining Triplet" row. The repo default, CE + Soft-Weighted Triplet + KL, is Table VII's 83.01 row.
- **Learning rate:** **0.003** for Global (the default in Section VI-A) and **0.01** for FSRA/LPN (Section VI-E: "the learning rate of the chunking-based method is set to 0.01").

Everything else is unchanged: ViT-S/224, bs 16, 120 epochs, LR x0.1 at 70/110, and the same augmentations (rotate UAV, affine + erasing satellite, which matches Table IV's 80.18 row).

**Settings:** Accelerator `GPU T4 x2`, Internet `On`, Input `netvoych1k/denseuav`. Use *Save & Run All* (about 6 h).
**Changes from the repo:** the same compatibility fixes as `baseline02.ipynb`, plus the loss and lr settings above. The loss and lr settings were inferred from the paper's tables because the repo does not ship Table VI configs, so expect about ±1 R@1 of run-to-run noise.
Params is about 0.37M below the paper. The likely reason is that newer thop skips timm's unused 1000-class ViT head. InferTime is measured on a T4.

In [1]:
import glob, os
ROOT = os.path.dirname(glob.glob('/kaggle/input/**/Dense_GPS_ALL.txt', recursive=True)[0])
# class counts from the repo README
for d, n in [('train/drone', 2256), ('train/satellite', 2256), ('test/query_drone', 777), ('test/gallery_satellite', 3033)]:
    assert len(os.listdir(f'{ROOT}/{d}')) == n, d
print(ROOT, '|', os.cpu_count(), 'vCPUs')
!nvidia-smi -L

/kaggle/input/datasets/netvoych1k/denseuav/DenseUAV | 4 vCPUs
GPU 0: Tesla T4 (UUID: GPU-2e7ae4b4-2e4a-6a36-435b-cd0783f1e4ec)
GPU 1: Tesla T4 (UUID: GPU-86db3b37-48e2-e544-4f36-3053cb012090)


In [2]:
%cd /kaggle/working
!git clone -q https://github.com/Dmmm1997/DenseUAV.git && git -C DenseUAV checkout -q b8de187
%cd /kaggle/working/DenseUAV
!pip install -q -r requirments.txt
!touch datasets/__init__.py losses/__init__.py optimizers/__init__.py tool/__init__.py
!sed -i 's/np\.bool8/np.bool_/g; s/np\.in1d/np.isin/g' evaluateDistance.py evaluate_gpu.py
!sed -i 's/timm\.models\.layers/timm.layers/' models/Backbone/cvt.py
!sed -i 's#^root_dir=.*#root_dir="{ROOT}"#; s#^num_worker=.*#num_worker={os.cpu_count()}#; s#^head_pool=.*#head_pool="global"#; s#^feature_loss=.*#feature_loss="HardMiningTripletLoss"#; s#^kl_loss=.*#kl_loss="no"#' train_test_local.sh
p = 'tool/get_inference_time.py'
src = open(p).read().replace('yaml.load(stream)', 'yaml.safe_load(stream)').replace(
    'print("inference_time = {}s".format(time.time()-since))',
    'print(f"Params={params} Macs={macs} InferTime={time.time()-since:.1f}s")')
assert 'safe_load' in src and 'Params=' in src
open(p, 'w').write(src)
!mkdir -p checkpoints && head -24 train_test_local.sh

/kaggle/working
/kaggle/working/DenseUAV
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 1.1 MB/s eta 0:00:00
name="baseline"
root_dir="/kaggle/input/datasets/netvoych1k/denseuav/DenseUAV"
data_dir=$root_dir/train
test_dir=$root_dir/test
gpu_ids=0
num_worker=4
lr=0.01
batchsize=16
sample_num=1
block=1
num_bottleneck=512
backbone="ViTS-224" # resnet50 ViTS-224 senet
head="SingleBranch"
head_pool="global"
cls_loss="CELoss" # CELoss FocalLoss
feature_loss="HardMiningTripletLoss"
kl_loss="no"
h=224
w=224
load_from="no"
ra="satellite"  # random affine
re="satellite"  # random erasing
cj="no"  # color jitter
rr="uav"  # random rotate


In [3]:
# (checkpoint name, head, block, lr, paper R@1, paper R@5) from Table VI
RUNS = [('Global', 'SingleBranch', 1, 0.003, 80.18, 93.99),
        ('FSRA_B2', 'FSRA', 2, 0.01, 82.58, 94.94),
        ('LPN_B2', 'LPN', 2, 0.01, 83.05, 94.89)]
for name, head, block, lr, *_ in RUNS:
    !sed -i 's/^name=.*/name="{name}"/; s/^head=.*/head="{head}"/; s/^block=.*/block={block}/; s/^lr=.*/lr={lr}/' train_test_local.sh
    !bash train_test_local.sh
    !cd checkpoints/{name}/tool && PYTHONPATH=.. python get_inference_time.py > ../infer.txt

/kaggle/working/DenseUAV/models/Backbone/cvt.py:553: SyntaxWarning: "is" with 'str' literal. Did you mean "=="?
  or pretrained_layers[0] is '*'
Namespace(gpu_ids='0', name='Global', data_dir='/kaggle/input/datasets/netvoych1k/denseuav/DenseUAV/train', num_worker=4, batchsize=16, pad=0, h=224, w=224, rr='uav', ra='satellite', re='satellite', cj='no', erasing_p=0.3, warm_epoch=0, lr=0.003, DA=False, droprate=0.5, autocast=True, block=1, cls_loss='CELoss', feature_loss='HardMiningTripletLoss', kl_loss='no', sample_num=1, num_epochs=120, num_bottleneck=512, load_from='no', backbone='ViTS-224', head='SingleBranch', head_pool='global')
[<datasets.queryDataset.RotateAndCrop object at 0x7fd095cb9160>, Resize(size=(224, 224), interpolation=bicubic, max_size=None, antialias=True), Pad(padding=0, fill=0, padding_mode=edge), RandomHorizontalFlip(p=0.5), ToTensor(), Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])]
[Resize(size=(224, 224), interpolation=bicubic, max_size=None, anti

In [4]:
import json, re
import pandas as pd
rows = []
for name, head, block, lr, p1, p5 in RUNS:
    B = f'checkpoints/{name}'
    r = dict(re.findall(r'([\w@]+)[:=]([\d.]+[MGs]?)', open(f'{B}/results.txt').read() + open(f'{B}/infer.txt').read()))
    sdm = json.load(open(f'{B}/SDM@K(1,100).json'))
    rows.append({'Head': name, 'Params': r['Params'], 'Macs': r['Macs'], 'InferTime': r['InferTime'],
                 'R@1': r['Recall@1'], 'R@5': r['Recall@5'], 'R@10': r['Recall@10'], 'AP': r['AP'],
                 'SDM@1': f"{sdm['1'] * 100:.2f}", 'paper R@1': p1, 'paper R@5': p5})
pd.DataFrame(rows).set_index('Head')

,Params,Macs,InferTime,R@1,R@5,R@10,AP,SDM@1,paper R@1,paper R@5
Head,,,,,,,,,,
Global,22.926M,8.484G,26.9s,79.45,93.52,97.00,69.10,83.92,80.18,93.99
FSRA_B2,25.637M,8.490G,29.6s,84.51,95.67,98.16,73.32,87.69,82.58,94.94
LPN_B2,25.637M,8.490G,30.1s,84.34,95.80,98.20,73.77,87.48,83.05,94.89
